In [1]:
import torch
import transformers
import peft
import bitsandbytes
import datasets
import accelerate
import pandas
import tokenizers
import huggingface_hub
from importlib.metadata import version, PackageNotFoundError

def get_version(package_name):
    try:
        module = __import__(package_name)
        return module.__version__
    except (AttributeError, ImportError):
        try:
            return version(package_name)
        except PackageNotFoundError:
            return "Not Installed"

libraries = {
    "torch": torch.__version__,
    "cuda_version": torch.version.cuda,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "bitsandbytes": get_version("bitsandbytes"),
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "tokenizers": tokenizers.__version__,
    "huggingface-hub": huggingface_hub.__version__,
    "pandas": pandas.__version__,
}

print("### Reporte para requirements.txt ###")
for lib, ver in libraries.items():
    print(f"{lib}=={ver}")


### Reporte para requirements.txt ###
torch==2.1.1+cu121
cuda_version==12.1
transformers==4.40.2
peft==0.10.0
bitsandbytes==0.41.3
datasets==2.18.0
accelerate==0.28.0
tokenizers==0.19.1
huggingface-hub==0.20.3
pandas==2.2.0


## Librerías y Parámetros



In [2]:
import torch
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import time
import pickle

LANGUAGE = "asturiano"
MODELO = "Qwen/Qwen2.5-7B-Instruct" 
PRELOADED = True
PRELOADED_PATH = "./qlora_asturiano__Qwen2.5-7B-Instruct_04-29_22-31-39/checkpoint-22500"

# Define output directory for QLoRA adapter
date = time.localtime(time.time())
output_dir = f"./qlora_{LANGUAGE}__{MODELO.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}"

print(f"Model name set to: {MODELO}")
print(f"Output directory set to: {output_dir}")

2026-04-29 22:31:37.659229: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-29 22:31:37.659298: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-29 22:31:37.660977: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-29 22:31:37.669890: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-29 22:31:38.963901: W tensorflow/compiler/tf2

Model name set to: Qwen/Qwen2.5-7B-Instruct
Output directory set to: ./qlora_asturiano__Qwen2.5-7B-Instruct_04-29_22-31-39


In [3]:
def loadQlora(base_model_hf_name, qlora_adapter_path):
    # 1. Define the 4-bit quantization configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load the base pre-trained language model with quantization
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_hf_name,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        tie_word_embeddings=False # Added to silence the warning
    )

    # 3. Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_hf_name, trust_remote_code=True)
    # Set pad_token_id if it's None, typically to eos_token_id for causal LMs
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # 4. Load the QLoRA adapter onto the base model
    peft_model = PeftModel.from_pretrained(base_model, qlora_adapter_path)

    print(f"Base model '{base_model_hf_name}' loaded with 4-bit quantization.")
    print(f"Tokenizer loaded.")
    print(f"QLoRA adapter loaded from '{qlora_adapter_path}'.")

    return peft_model, tokenizer

print("loadQlora function defined.")

loadQlora function defined.


## Cuantización del Modelo

Cargar un modelo pre-entrenado de Hugging Face y configurarlo para la cuantización de 4 bits, preparándolo para QLoRA.


In [4]:
# 1. Define the 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 2. Load the pre-trained language model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODELO,
    quantization_config=bnb_config,
    trust_remote_code=True,
    tie_word_embeddings=False # Added to silence the warning about tied weights
)

# 3. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODELO, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<pad>"})
    model.resize_token_embeddings(len(tokenizer))

model.config.use_cache = False  # importante con Trainer
model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Model loaded with 4-bit quantization: {model.__class__.__name__}")
print(f"Model device: {model.device}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0


## Configuración QLoRA

Definir los hiperparámetros de QLoRA (por ejemplo, lora_r, lora_alpha, lora_dropout) y crear el modelo PEFT usando get_peft_model para envolver el modelo base cuantificado.


In [5]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)

peft_model = get_peft_model(model, lora_config)

print("QLoRA configuration applied.")
peft_model.print_trainable_parameters()

QLoRA configuration applied.
trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.26434798934534914


In [6]:
peft_model.active_peft_config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='Qwen/Qwen2.5-7B-Instruct', revision=None, task_type='CAUSAL_LM', inference_mode=False, r=8, target_modules={'v_proj', 'k_proj', 'q_proj', 'o_proj', 'down_proj', 'gate_proj', 'up_proj'}, lora_alpha=16, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None)

## Entrenamiento



In [7]:
import torch
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   7948 MiB |   8911 MiB |  22907 MiB |  14958 MiB |
|       from large pool |   7807 MiB |   8847 MiB |  22714 MiB |  14907 MiB |
|       from small pool |    141 MiB |    141 MiB |    192 MiB |     50 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   7948 MiB |   8911 MiB |  22907 MiB |  14958 MiB |
|       from large pool |   7807 MiB |   8847 MiB |  22714 MiB |

In [8]:
with open(f"TrainDatasets/{MODELO.split('/')[-1]}/{LANGUAGE}.pkl", "rb") as f:
    train, test = pickle.load(f)
train = train.shuffle()

print("Dataset real cargado y tokenizado correctamente.")
print(f"Ejemplo tokenizado: {len(train[2]['input_ids'])}")
print(len(train), len(test))

Dataset real cargado y tokenizado correctamente.
Ejemplo tokenizado: 156
358667 18878


In [9]:
from transformers import TrainingArguments, Trainer

# 4. Configure TrainingArguments
training_args = TrainingArguments(
    output_dir=output_dir, # Use the output_dir defined previously
    per_device_train_batch_size=1, # Small batch size for mock data
    num_train_epochs=3, # Small number of epochs for quick demonstration
    learning_rate=1e-4,
    logging_steps=100, # Log every step
    fp16=True,
    bf16=False,
    do_eval=False,
    save_strategy="steps",
    save_steps=5000,
    report_to='none', # Do not report to any service
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, # Prueba recomendacion cop
    gradient_accumulation_steps=2,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
)

# 5. Create an instance of Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train,
    # eval_dataset=test # Puesto a False 
)

# 6. Start the training process
if PRELOADED:
    trainer.train(resume_from_checkpoint=PRELOADED_PATH)
else:
    trainer.train()

print("QLoRA training completed.")

	save_steps: 2500 (from args) != 1500 (from trainer_state.json)


Step,Training Loss
12100,2.034300
12200,2.026200
12300,2.003600
12400,2.071600
12500,2.038200
12600,2.069800
12700,2.070100
12800,1.999400
12900,2.036200
13000,2.073100


: 

### Exportar


In [ ]:
# peft_model.save_pretrained(output_dir)
trainer.model.save_pretrained(output_dir)
print(f"QLoRA adapter saved to {output_dir}")

## Interacción con el Modelo Afinado


In [ ]:
def generate_text(prompt, model, tokenizer, max_length=100):
    # Tokenize the input prompt
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True)

    # Move inputs to the model's device (CPU in this case)
    input_ids = inputs['input_ids'].to(model.device)
    attention_mask = inputs['attention_mask'].to(model.device)

    # Generate text
    output_sequences = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        do_sample=True, # Enable sampling for more diverse outputs
        top_k=50, # Consider top 50 tokens for sampling
        top_p=0.95, # Nucleus sampling
        temperature=0.7, # Controls randomness
        pad_token_id=tokenizer.eos_token_id, # Ensure generation stops properly
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode the generated sequence
    generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=True)

    return generated_text


In [ ]:
print("\n--- Reloading and interacting with the fine-tuned model using loadQlora ---")

# Use the loadQlora function to load the PEFT model and tokenizer
# MODELO and output_dir are defined in previous cells
peft_finetuned_model, peft_finetuned_tokenizer = m,t#loadQlora(MODELO, output_dir)

# Now, use the loaded peft_finetuned_model and peft_finetuned_tokenizer with the generate_text function

prompt1 = "Hola, ¿cómo tas güei?"
response1 = generate_text(prompt1, peft_finetuned_model, peft_finetuned_tokenizer)
print(f"Prompt: {prompt1}")
print(f"Response: {response1}")

prompt2 = "¿Cuál ye'l to color preferíu?"
response2 = generate_text(prompt2, peft_finetuned_model, peft_finetuned_tokenizer)
print(f"\nPrompt: {prompt2}")
print(f"Response: {response2}")

print("Model interaction section updated to use loadQlora function.")